# Shailaputri — anomaly detection on real metric series

Model 1 of the nine. **Data:** NAB real series (MIT). **Method:** strictly causal features (rolling robust z at three windows, change, volatility, same time yesterday) into LightGBM. **Test:** leave one *family* of series out, scored online by the same harness as four statistical detectors.

Results: `docs/research/models/shailaputri.md`; decision: `docs/adr/SatiDetect.md`.

In [ ]:
# Run on Kaggle or Colab. Clones this repository and installs the two packages.
import os, subprocess, sys
if not os.path.exists("PASHUPATASTRA"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/preetraval45/PASHUPATASTRA.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "PASHUPATASTRA/packages/core", "-e", "PASHUPATASTRA/packages/sati", "matplotlib"], check=True)
sys.path[:0] = ["PASHUPATASTRA/packages/core", "PASHUPATASTRA/packages/sati"]

In [ ]:
from pathlib import Path
from sati.sources import nab
from sati.detect import features as feats
cache = Path("cache/nab")
rev = "ea702d75cc2258d9d7dd35ca8e5e2539d71f3140"
wins = nab.windows(cache, rev)
name = "realKnownCause/machine_temperature_system_failure.csv"
frame = nab.series(name, cache, rev)
ranges = nab.labelled_ranges(frame, wins[name])
ax = frame.plot(x="timestamp", y="value", figsize=(12, 3), legend=False, title=name)
for s, e in ranges: ax.axvspan(frame.timestamp[s], frame.timestamp[e - 1], alpha=0.2, color="red")

## Causal features — the no-look-ahead property

Changing the future must never change a past feature; `tests/testdetect.py` proves it, and so does this cell.

In [ ]:
import numpy as np
x = frame.value.to_numpy()
a = feats.features(x)
b = x.copy(); b[3000:] += 1000
assert np.allclose(a.iloc[:3000].to_numpy(), feats.features(b).iloc[:3000].to_numpy(), equal_nan=True)
a.describe().T

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "sati.detect.run", "--cache", "cache/nab", "--out", "results/shailaputri", "--revision", rev], check=True, cwd="PASHUPATASTRA/packages/sati")
import json, pandas as pd
r = json.load(open("PASHUPATASTRA/packages/sati/results/shailaputri/results.json"))
pd.DataFrame(r["mean_across_families"]).T